## YOLO Notebook Version

Notebook equivalent of `yolo.py` for training/evaluation experiments.


### Helper methods

In [2]:
import json
import shutil

from pathlib import Path

import cv2
from ultralytics import YOLO

# Resolve repo root whether notebook runs from repo root or yolo/ directory
CWD = Path.cwd().resolve()
REPO_ROOT = CWD.parent if CWD.name == "yolo" else CWD

In [3]:

# All predefined helper functions from yolo.py
def tune_model():
    # Load a pretrained YOLO model (you can choose n, s, m, l, or x versions)
    model = YOLO("yolo11n.pt")

    # Start training on your custom dataset
    data_yaml = REPO_ROOT / "ingredients_data" / "data.yaml"

    model.tune(
        data=str(data_yaml),
        epochs=50,  # epochs per trial
        iterations=10,  # number of tuning trials
        imgsz=800,
        batch=16,
        optimizer="AdamW",
        project=str(REPO_ROOT / "yolo" / "runs"),
        name="ingredients_tune11n",
    )

def train_model(epochs, imgsz, batch, patience, lr0, lrf, box, cls, weight_decay):
    """
    More complex training function with finetuning and specific hyperparam definition
    """
    data_yaml = REPO_ROOT / "ingredients_data" / "data.yaml"

    model = YOLO("yolo11n.pt")

    model.train(
        data=str(data_yaml),
        epochs=epochs,
        imgsz=imgsz,
        batch=batch,
        patience=patience,
        optimizer="AdamW",
        lr0=lr0,
        lrf=lrf,
        box=box,
        cls=cls,
        weight_decay=weight_decay,
        multi_scale=True,
        mosaic=1.0,
        copy_paste=0.4,
        scale=0.9,
        translate=0.1,
        degrees=5.0,
        shear=2.0,
        perspective=0.0005,
        close_mosaic=10,
        project=str(REPO_ROOT / "yolo" / "runs"),
        name="yolo11n_fridge",
    )


def test_model(model, dataset_path, conf=0.25, iou=0.6):
    """
    Run object detection and export:
    1) images with YOLO bounding boxes
    2) per-detection crops for downstream DINO classification
    """
    source_path = Path(dataset_path)
    if not source_path.is_absolute():
        source_path = (REPO_ROOT / source_path).resolve()

    output_root = REPO_ROOT / "yolo" / "runs" / "eval"  # clean this up maybe
    crops_root = output_root / "crops"
    if crops_root.exists():
        shutil.rmtree(crops_root)
    crops_root.mkdir(parents=True, exist_ok=True)

    results = model.predict(
        source=str(source_path),
        conf=conf,
        iou=iou,
        save=True,
        save_crop=False,
        project=str(output_root),
        name="predictions",
        exist_ok=True,
    )

    pred_ingredients = {}

    for image_idx, result in enumerate(results):
        best_pred_per_ingredient = {}
        image_name = Path(result.path).stem if result.path else f"image_{image_idx}"
        image = result.orig_img

        for box_idx, box in enumerate(result.boxes):
            class_id = int(box.cls.item())
            ingredient = model.names[class_id]
            confidence = float(box.conf.item())
            x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())

            if ingredient not in best_pred_per_ingredient or confidence > best_pred_per_ingredient[ingredient]:
                best_pred_per_ingredient[ingredient] = confidence

            crop = image[y1:y2, x1:x2].copy()
            if crop.size == 0:
                continue

            crop_path = crops_root / f"{image_name}_{box_idx}.jpg"
            cv2.imwrite(str(crop_path), crop)

        ingredient_list = [
            {"ingredient": cls, "confidence": round(conf, 3)} for cls, conf in best_pred_per_ingredient.items()
        ]
        print(f"{Path(result.path).name}: {ingredient_list}")
        pred_ingredients[f"{Path(result.path).name}"] = ingredient_list

    print(f"Prepared crops for DINO under {crops_root}")
    return pred_ingredients


def evaluate_models(model_entries, data_yaml):
    """
    Evaluate multiple YOLO checkpoints and prints a compact comparison table for writeup
    """

    rows = []
    for label, weights_path in model_entries:
        path = Path(weights_path)
        if not path.exists():
            print(f"[SKIP] {label}: checkpoint not found at {path}")
            continue

        model = YOLO(str(path))
        metrics = model.val(data=str(data_yaml), verbose=False, project=str(REPO_ROOT / "yolo" / "runs"))
        row = {
            "label": label,
            "precision": float(metrics.box.mp),
            "recall": float(metrics.box.mr),
            "mAP50": float(metrics.box.map50),
            "mAP50_95": float(metrics.box.map),
        }
        rows.append(row)

    print("\nModel comparison (higher better):")
    print(f"{'Label':<35} {'Precision':>10} {'Recall':>10} {'mAP50':>10} {'mAP50_95':>10}")
    print("-" * 80)
    for row in rows:
        print(
            f"{row['label']:<35} "
            f"{row['precision']:>10.4f} "
            f"{row['recall']:>10.4f} "
            f"{row['mAP50']:>10.4f} "
            f"{row['mAP50_95']:>10.4f}"
        )


def inference_sweep(model, data_yaml="ingredients_data/data.yaml"):
    confs = [0.10, 0.25, 0.40, 0.55]
    ious = [0.50, 0.60, 0.70]
    for conf in confs:
        for iou in ious:
            metrics = model.val(data=data_yaml, conf=conf, iou=iou)
            print(f"conf={conf:.2f}, iou={iou:.2f}")
            print(metrics.box)


def parse_json(jsonl_path):
    records = {}
    with open(jsonl_path, encoding="utf-8") as f:
        for line in f:
            row = json.loads(line)
            records[Path(row["image"]).name] = row.get("true_ingredients", [])
    return records


def classification_accuracy(pred_ingredients, actual_ingredients):
    """
    Given detected ingredients and ground truth ingredients,
    compute per-image precision/recall/F1 and macro averages.
    """
    per_image = {}
    total_tp = total_fp = total_fn = 0

    for image_name, preds in pred_ingredients.items():
        pred_set = {p["ingredient"].lower() for p in preds}
        print(f"Predicted set: {pred_set}")
        true_set = set(i.lower() for i in actual_ingredients.get(image_name, []))
        print(f"True set: {true_set}")

        tp = len(pred_set & true_set)
        fp = len(pred_set - true_set)
        fn = len(true_set - pred_set)

        precision = tp / (tp + fp) if (tp + fp) else 0.0
        recall = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0

        per_image[image_name] = {
            "tp": tp,
            "fp": fp,
            "fn": fn,
            "precision": round(precision, 4),
            "recall": round(recall, 4),
            "f1": round(f1, 4),
        }
        total_tp += tp
        total_fp += fp
        total_fn += fn

    global_precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) else 0.0
    global_recall = total_tp / (total_tp + total_fn) if (total_tp + total_fn) else 0.0
    global_f1 = (
        2 * global_precision * global_recall / (global_precision + global_recall)
        if (global_precision + global_recall)
        else 0.0
    )

    summary = {
        "global_precision": round(global_precision, 4),
        "global_recall": round(global_recall, 4),
        "global_f1": round(global_f1, 4),
        "images_evaluated": len(per_image),
    }

    return {"summary": summary, "per_image": per_image}

### RUNNING YOLO VERSION 

In [ ]:
"""
If model has not been trained, then run this cell. Otherwise, use trained models and just run the comparison metrics below.
"""
default_model = train_model(
    epochs=50,
    imgsz=896,
    batch=16, 
    patience=25,
    lr0=0.001,
    lrf=0.01,
    box=8.0,
    cls=0.5,
    weight_decay=0.0005
)

In [4]:
# Comparing different tuned models 
data_yaml = REPO_ROOT / "ingredients_data" / "data.yaml"

model_entries = [
    ("yolo11n baseline", REPO_ROOT / "yolo_runs" / "ingredients_yolo11n" / "weights" / "best.pt"),
    ("yolo11n baseline (run 2)", REPO_ROOT / "yolo_runs" / "ingredients_yolo11n-2" / "weights" / "best.pt"),
    ("yolo11s baseline", REPO_ROOT / "yolo_runs" / "ingredients_yolo11s" / "weights" / "best.pt"),
    ("yolo11n tuned", REPO_ROOT / "yolo_runs" / "ingredients_tune11n" / "weights" / "best.pt"),
    ("yolo11n tuned", REPO_ROOT / "yolo"/ "runs" / "yolo11n_fridge" / "weights" / "best.pt"),
    ("yolo11n tuned", REPO_ROOT / "yolo"/ "runs" / "yolo11n_fridge-2" / "weights" / "best.pt")

]

evaluate_models(model_entries, data_yaml)

[SKIP] yolo11n baseline: checkpoint not found at /oscar/data/class/csci1430/students/sli347/cv-final/yolo_runs/ingredients_yolo11n/weights/best.pt
Ultralytics 8.4.41 🚀 Python-3.9.21 torch-2.8.0+cu128 CPU (Intel Xeon Platinum 8268 CPU @ 2.90GHz)
YOLO11n summary (fused): 101 layers, 2,651,527 parameters, 0 gradients, 6.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 29.8±6.1 MB/s, size: 60.8 KB)
val: Scanning /oscar/data/class/csci1430/students/sli347/cv-final/ingredients_data/valid/labels.cache... 824 images, 5 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 824/824 27.9Mit/s 0.0s
WARNING ⚠️ Box and segment counts should be equal, but got len(segments) = 60, len(boxes) = 1985. To resolve this only boxes will be used and all segments will be removed. To avoid this please supply either a detect or segment dataset, not a detect-segment mixed dataset.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 73% ━━━━━━━━╸─── 38/52 7.4s/it 2:56<1:445


KeyboardInterrupt: 

In [7]:
# Qualitative metrics for best model 
print("\nQualitative Metrics")
best_checkpoint = REPO_ROOT / "yolo"/ "runs" / "yolo11n_fridge-2" / "weights" / "best.pt"
pred_ingredients = test_model(YOLO(str(best_checkpoint)), dataset_path=REPO_ROOT / "eval_data" / "images", iou=0.75)


Qualitative Metrics

image 1/33 /oscar/data/class/csci1430/students/sli347/cv-final/eval_data/images/fridge_test1.jpg: 672x896 2 Bananas, 1 Broccoli, 291.3ms
image 2/33 /oscar/data/class/csci1430/students/sli347/cv-final/eval_data/images/fridge_test2.jpg: 480x896 2 Eggs, 1 Turnip, 381.5ms
image 3/33 /oscar/data/class/csci1430/students/sli347/cv-final/eval_data/images/fridge_test3.jpg: 896x896 (no detections), 478.9ms
image 4/33 /oscar/data/class/csci1430/students/sli347/cv-final/eval_data/images/fridge_test4.jpg: 896x608 1 Egg, 145.5ms
image 5/33 /oscar/data/class/csci1430/students/sli347/cv-final/eval_data/images/fridge_test5.jpg: 512x896 (no detections), 185.0ms
image 6/33 /oscar/data/class/csci1430/students/sli347/cv-final/eval_data/images/photo-1536181211993-cf4b2c100475.jpg: 896x608 (no detections), 180.9ms
image 7/33 /oscar/data/class/csci1430/students/sli347/cv-final/eval_data/images/photo-1563865436874-9aef32095fad.jpg: 896x608 (no detections), 265.1ms
image 8/33 /oscar/data/c

In [8]:
actual_ingredients = parse_json(REPO_ROOT / "eval_data" / "labels.jsonl")
metrics = classification_accuracy(pred_ingredients, actual_ingredients)
metrics["summary"]

Predicted set: {'banana', 'broccoli'}
True set: {'broccoli', 'banana', 'tomato', 'ham', 'cheese', 'avocado'}
Predicted set: {'turnip', 'egg'}
True set: {'cauliflower', 'carrot', 'parsley', 'red onion', 'red cabbage', 'tomato', 'cucumber', 'kale', 'cabbage', 'egg'}
Predicted set: set()
True set: {'bell pepper', 'carrot', 'tomato', 'cucumber', 'bread', 'yogurt', 'cherry tomato', 'egg', 'lettuce', 'apple', 'milk'}
Predicted set: {'egg'}
True set: {'bell pepper', 'carrot', 'blueberry', 'grape', 'cucumber', 'almond milk', 'bread', 'scallion', 'spinach', 'egg', 'lettuce', 'mushroom', 'avocado'}
Predicted set: set()
True set: {'bell pepper', 'broccoli', 'cauliflower', 'butter', 'carrot', 'tomato', 'cucumber', 'bread', 'cabbage', 'lettuce', 'milk'}
Predicted set: set()
True set: {'egg', 'milk'}
Predicted set: set()
True set: {'peach', 'onion', 'fig', 'tomato', 'cucumber', 'garlic', 'cherry tomato', 'walnut', 'apple', 'potato'}
Predicted set: set()
True set: {'raspberry', 'bell pepper', 'strawb

{'global_precision': 0.4688,
 'global_recall': 0.0484,
 'global_f1': 0.0877,
 'images_evaluated': 33}